# Week 4, Day 5: Port a title-history simulator to Rust

**Lesson:** [week4/day5.ipynb](../../../week4/day5.ipynb), *Code generator: results*. **About 45 minutes.**

**What you practice here**

- Asking a model for the toolchain commands, with your machine's details in the prompt: one user message in a Chat Completions call, as the lesson does for Rust.
- Writing the porting prompts once for any target language, C++ or Rust, as the lesson's prompts on `{language}` do.
- A compile-and-run function that an app's button can call: it writes the file, compiles and runs it with `subprocess.run`, and returns the output, or the error as text.
- The full three-button Gradio app, used on a program where careless ports go wrong, in Try it for real.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week4/day5", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson adds Rust beside C++, gives a harder program, `python_hard`, and wraps everything in an app with three
buttons: port the code, run the Python, and run the port. Six of Ed's nine models fail on it: their programs print
the wrong result. Do the same here for a championship title-history simulator: 50 million matches, in which a 3%
upset changes the title. Its generator works modulo 2**48, so a port that is careless with 64-bit integers prints the
wrong numbers, as careless ports of `python_hard` did.

You write the question that asks a model how to compile Rust on your machine, the porting prompts for any language,
and the function behind the app's Run button. The two functions that reach outside take what they call as their first
parameter: `ask_for_commands` takes the client and makes the Chat Completions call itself, and `compile_and_run`
takes `subprocess.run` as `run` and calls it as you would call `subprocess.run`. The checks hand them stand-ins
shaped like the real ones: they record what you send, so no model, compiler or program runs.

## What each call should give

```python
display(Markdown(ask_for_commands(client, retrieve_system_info(), rust_toolchain_info())))
# makes one request with the model gpt-5 and one user message that holds both reports, and returns the reply's
# markdown, such as 'You already have rustc 1.90 ... compile_command = ["rustc", "main.rs", "-C", "opt-level=3", ...]'

messages_for(TITLES, "Rust", system_info, rust_compile)
# [{'role': 'system', 'content': 'Your task is to convert Python code into high-performance Rust code. ...'},
#  {'role': 'user', 'content': 'Port this Python code to Rust ... written to a file called main.rs ...'}]

compile_and_run(subprocess.run, rust_code, "/tmp/tmpq1w2/main.rs", rust_compile, ["/tmp/tmpq1w2/main"])
# writes main.rs, compiles it, runs it once, and returns what it printed:
# 'Title changes: ...\nLongest reign: ... matches\n...'
# or, when a step fails: 'An error occurred:\nerror[E0308]: mismatched types ...'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import subprocess

## Given

The program you port: a championship title-history simulator, as the lesson gives `python_hard`. Its generator works
modulo 2**48, so a port must keep those large numbers exact. Run this cell: Try it for real uses `TITLES`, and the
parts don't need it.

In [ ]:
TITLES = """# Be careful to support large numbers: the generator works modulo 2**48
import time

def next_state(state):
    return (25214903917 * state + 11) % 2**48

def title_history(matches, roster, seed):
    state = seed
    champion, changes, reign, longest = 0, 0, 0, 0
    for _ in range(matches):
        state = next_state(state)
        challenger = (state >> 17) % roster
        state = next_state(state)
        if challenger != champion and (state >> 17) % 100 < 3:  # a 3% upset: the title changes hands
            champion, changes, reign = challenger, changes + 1, 0
        else:
            reign += 1
            longest = max(longest, reign)
    return changes, longest, champion

start_time = time.time()
changes, longest, champion = title_history(50_000_000, 40, 1963)
end_time = time.time()

print("Title changes:", changes)
print("Longest reign:", longest, "matches")
print("Champion at the end: roster number", champion)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

### 1 · `ask_for_commands(client, system_info, toolchain_info, language="Rust", model="gpt-5")`

Asks the model how to compile and run one file on your machine, and returns its answer. `system_info` is the dict the
lesson's `retrieve_system_info()` returns, and `toolchain_info` is the toolchain's dict: for Rust,
`rust_toolchain_info()`. Make one Chat Completions call with the model `model` and one user message, as the lesson
does when it asks about Rust. The message:

- holds `system_info` and `toolchain_info` as text;
- names `language` and the file to compile: `main.rs` for `"Rust"`, `main.cpp` for `"C++"`;
- asks whether anything must be installed, and for the exact `compile_command` and `run_command` for the fastest
  runtime, with those two names in it.

Return the reply's text as it is: it is markdown, for `display(Markdown(...))`.

<details><summary>Hint</summary>

```python
messages = [{"role": "user", "content": question}]
response = client.chat.completions.create(model=model, messages=messages)
return response.choices[0].message.content
```

</details>

In [ ]:
def ask_for_commands(client, system_info, toolchain_info, language="Rust", model="gpt-5"):
    # Write ask_for_commands here, as described above.
    raise NotImplementedError("ask_for_commands is not written yet")

In [ ]:
check("ask_for_commands")

### 2 · `messages_for(python, language, system_info, compile_command)`

Builds the porting messages for any target language, as the lesson's prompts on `{language}` do. `language` is
`"C++"` or `"Rust"`, and the other parameters are as in Day 3's `messages_for`. Return a new list of two messages, a
`"system"` one, then a `"user"` one:

- the system message names `language` and no other language: a Rust prompt must not mention C++, nor a C++ prompt
  Rust. As on Day 3, it asks for `language` code only, and for output identical to the Python program's in the
  fastest time, and it uses the words "only" and "identical", which the checks look for;
- the user message names `language`, says the reply is written to `main.rs` for Rust or `main.cpp` for C++, and holds
  the system info as text, every item of `compile_command`, and the Python code in a fenced block: a line of three
  backticks followed by `python`, the code, then a line of three backticks.

The lesson's user prompt says the file is `main.{language}`, which would be `main.Rust`: the file is
`main.{extension}`.

<details><summary>Hint</summary>

Day 3's prompts, as f-strings on `language`:

```python
system = f"""Your task is to convert Python code into high-performance {language} code.
Respond only with {language} code ... identical output ..."""
filename = "main.rs" if language == "Rust" else "main.cpp"
```

</details>

In [ ]:
def messages_for(python, language, system_info, compile_command):
    # Write messages_for here, as described above.
    raise NotImplementedError("messages_for is not written yet")

In [ ]:
check("messages_for")

### 3 · `compile_and_run(run, code, filename, compile_command, run_command)`

Writes the code to its file, compiles it, runs it once, and returns what it printed, or the error as text. It is the
lesson's `compile_and_run`, which the app's Run button calls. `run` is `subprocess.run` (the checks pass a stand-in
shaped like it), `filename` is the source file's full path, and the two commands are lists of strings. In order:

- write `code` to `filename`, as UTF-8;
- run `compile_command`, then `run_command`, each through `run` with the command first, asking it to raise an error
  when the command fails, to capture what the command prints, and to give that back as text;
- return what `run_command` printed.

If a step fails, return `"An error occurred:\n"` followed by the error output of the command that failed, as the
lesson does, and run nothing after it. The app then shows the compiler's complaint, or the program's crash, in place
of a result.

<details><summary>Hint</summary>

```python
run(compile_command, check=True, text=True, capture_output=True)
result = run(run_command, check=True, text=True, capture_output=True)
return result.stdout
```

A failed command raises `subprocess.CalledProcessError`, whose `.stderr` holds the error output.

</details>

In [ ]:
def compile_and_run(run, code, filename, compile_command, run_command):
    # Write compile_and_run here, as described above.
    raise NotImplementedError("compile_and_run is not written yet")

In [ ]:
check("compile_and_run")

## Check everything

The cell below runs all 7 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 4, Day 5 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A port with `gpt-5-nano` costs about a cent or less, and `openai/gpt-oss-120b`, the winner of the lesson's hard
program, runs on Groq, which has a free tier. Compiling needs Rust's `rustc`: the answer in step 2 says what to
install. If you install Rust now, this notebook's kernel won't find `rustc` on its `PATH` until you restart it, so the
solution also looks in `~/.cargo/bin`. Running the Python takes about 20 seconds.

1. Load your keys with `load_dotenv(override=True)`. Put the course's `week4` folder on the path with
   `sys.path.append(str(ROOT / "week4"))`, and import `retrieve_system_info` and `rust_toolchain_info` from
   `system_info`, and `CSS` from `styles`. Load Days 3 and 4 with `functions_of`, and build a `clients` dict from model
   name to client, as on Day 4: `gpt-5-nano`, and `openai/gpt-oss-120b` on Groq if you have its key.
2. Set `system_info = retrieve_system_info()`, ask a cheap model for your Rust commands with `ask_for_commands`,
   passing `rust_toolchain_info()` too, and show its answer with `display(Markdown(...))`.
3. Find `rustc` with `shutil.which("rustc")`, or at `~/.cargo/bin/rustc` if you installed Rust after starting the
   kernel. Write `rust_compile` and `run_command` from the answer, with the source file, `source`, and the program in
   a folder from `tempfile.mkdtemp()`. Then write the button functions: `convert(model, python)` returns
   `day4.port_with(clients, model, messages_for(python, "Rust", system_info, rust_compile))`, and `run_rust(code)`
   returns `compile_and_run(subprocess.run, code, source, rust_compile, run_command)`.
4. Build the lesson's app with `CSS` and `gr.themes.Monochrome()`: the Python pane holding `TITLES`, the Rust pane,
   the dropdown, the three buttons with the lesson's `elem_classes` (Run Python calls `day4.run_python`), and the two
   result areas. Launch it.
5. Port `TITLES` with two models and run both versions. Note which ports print the same three result lines as
   Python, and how fast the correct ones are.

<details><summary>Stuck? One way to write it</summary>

```python
import os
import shutil
import subprocess
import sys
import tempfile

import gradio as gr
from dotenv import load_dotenv
from IPython.display import Markdown, display

from check import functions_of

load_dotenv(override=True)
sys.path.append(str(ROOT / "week4"))  # the course's week4 folder: system_info.py and styles.py
from styles import CSS
from system_info import retrieve_system_info, rust_toolchain_info

day3, day4 = functions_of("week4/day3"), functions_of("week4/day4")
providers = {**day3.frontier_clients(os.environ), **day4.open_source_clients(os.environ)}
clients = {"gpt-5-nano": providers["openai"]}
if "groq" in providers:  # the winner of the lesson's hard program
    clients["openai/gpt-oss-120b"] = providers["groq"]

system_info = retrieve_system_info()
display(Markdown(ask_for_commands(providers["openai"], system_info, rust_toolchain_info(), model="gpt-5-nano")))
```

```python
# Installed Rust after starting the kernel? Its PATH lacks ~/.cargo/bin until a restart, so look there too
rustc = shutil.which("rustc") or os.path.expanduser("~/.cargo/bin/rustc")
folder = tempfile.mkdtemp()
source, program = os.path.join(folder, "main.rs"), os.path.join(folder, "main")
rust_compile = [rustc, source, "-C", "opt-level=3", "-C", "target-cpu=native", "-o", program]  # from the answer
run_command = [program]


def convert(model, python):
    return day4.port_with(clients, model, messages_for(python, "Rust", system_info, rust_compile))


def run_rust(code):
    return compile_and_run(subprocess.run, code, source, rust_compile, run_command)


with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Port from Python to Rust") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(label="Python (original)", value=TITLES, language="python", lines=26)
        with gr.Column(scale=6):
            rust = gr.Code(label="Rust (generated)", lines=26)
    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(list(clients), value=next(iter(clients)), show_label=False)
        port_button = gr.Button("Port to Rust", elem_classes=["convert-btn"])
        rust_run = gr.Button("Run Rust", elem_classes=["run-btn", "cpp"])
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            rust_out = gr.TextArea(label="Rust result", lines=8, elem_classes=["cpp-out"])
    port_button.click(fn=convert, inputs=[model, python], outputs=[rust])
    python_run.click(fn=day4.run_python, inputs=[python], outputs=[python_out])
    rust_run.click(fn=run_rust, inputs=[rust], outputs=[rust_out])

ui.launch(inbrowser=True)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.